In [1]:
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()
import sys
sys.path.append("../")
from dotenv import load_dotenv
load_dotenv()

from src.llm_utils import LLM_wrapper, RepoAssessment
from src.llm_utils import REPO_ASSESSMENT_PROMPT as repo_assessment_prompt
from src.repo_utils import RepoStatus

In [2]:
MODEL_NAME = "gpt-5.2-2025-12-11"
CONTEXT_WINDOW = 395000 # gpt 5.2's context window is 400k tokens

In [3]:
llm_wrapper = LLM_wrapper(
    model_name=MODEL_NAME,
    system_prompt=repo_assessment_prompt,
    output_format_class=RepoAssessment
)

In [4]:
df_pred = llm_wrapper.assess_dataframe(
    input_file_path="../data/03_repo_assessment/references_with_repo.parquet.br",
    text_column="repo_content",
    output_dir="../data/03_repo_assessment/",
    # we only want to process rows that were properly cloned
    row_filter=lambda row: row["repo_status"] == RepoStatus.OK
)

Loading checkpoint from ../data/03_repo_assessment/checkpoints/20260114_1614_gpt-5.2-2025-12-11.checkpoint.csv
Total rows: 447, Already processed: 100, Remaining (after filter): 280


100%|██████████| 280/280 [37:03<00:00,  7.94s/it]

Processing complete. Final results saved.


In [8]:
import ast

def normalize_coding_languages(x):
    # Nulls
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None

    # Already a list
    if isinstance(x, list):
        out = []
        for v in x:
            if v is None or (isinstance(v, float) and pd.isna(v)):
                continue
            s = str(v).strip().lower()
            if s:
                out.append(s)
        return out or None

    # Strings
    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return None

        # Case 1: a real Python literal string like "['python', 'r']"
        if s.startswith("[") and s.endswith("]") and "'" in s:
            try:
                val = ast.literal_eval(s)
                if isinstance(val, list):
                    return normalize_coding_languages(val)
            except Exception:
                pass

        # Case 2: bracketed but not quoted, like "[python, r]" or "[r]"
        if s.startswith("[") and s.endswith("]"):
            inner = s[1:-1].strip()
            if inner == "":
                return None
            parts = [p.strip().strip("'").strip('"').lower() for p in inner.split(",")]
            parts = [p for p in parts if p]
            return parts or None

        # Otherwise treat as single label
        return [s.lower()]

    # Fallback
    return [str(x).strip().lower()]

df_pred["coding_languages"] = df_pred["coding_languages"].apply(normalize_coding_languages)

In [9]:
df_pred["coding_languages"].value_counts()

coding_languages
[r]                                      123
[python]                                  98
[python, jupyter notebook]                14
[python, r]                               13
[python, r, shell]                         5
                                        ... 
[r, c, c/c++ headers]                      1
[matlab, c++]                              1
[python, shell, yaml, markdown, toml]      1
[r, sql, json]                             1
[json]                                     1
Name: count, Length: 82, dtype: int64

In [10]:
df_pred.to_parquet("../data/03_repo_assessment/references_repo_assessment_pred.parquet.br", index=False, compression="brotli")